# Task 1: Quantum Entanglement and Measurement Statistics

**Solution Notebook - Qiskit 1.x**

## Learning Objectives

- Create quantum circuits using Qiskit 1.x
- Implement Hadamard and CNOT gates
- Create and analyse Bell states (entangled states)
- Perform quantum measurements and interpret results
- Analyse measurement statistics and quantum correlations
- Visualise probability distributions

## Prerequisites

- Basic Python programming
- Understanding of probability

## Estimated Time: 45-60 minutes

In [ ]:
# Google Colab Setup - Run this cell first if using Colab
import sys
if 'google.colab' in sys.modules:
    print("📦 Installing dependencies for Google Colab...")
    !pip install -q qiskit>=1.0.0 qiskit-aer>=0.13.0 matplotlib seaborn pylatexenc
    print("✓ Dependencies installed successfully!")
    print("You can now run the rest of the notebook.\n")
else:
    print("✓ Running in local environment")

## Setup and Imports

**Important:** This notebook uses Qiskit 1.x API, which differs from pre-1.0 versions.

In [ ]:
# SOLUTION: Import required libraries for Qiskit 1.x
import numpy as np
import matplotlib.pyplot as plt

# Qiskit 1.x imports
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_distribution

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Print versions
import qiskit
import qiskit_aer
print(f"Qiskit version: {qiskit.__version__}")
print(f"Qiskit Aer version: {qiskit_aer.__version__}")
print("\nSetup complete! ✓")

### Qiskit 1.x Migration Notes

**Key Changes from Pre-1.0:**
- Import `AerSimulator` from `qiskit_aer` (not `qiskit.Aer`)
- Use `AerSimulator()` instead of `Aer.get_backend('qasm_simulator')`
- Use `transpile() + simulator.run()` instead of `execute()`
- No need for `assemble()` in Qiskit 1.x

---

## Exercise 1: Create a Bell State Circuit

### Background

A **Bell state** is a maximally entangled two-qubit state:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

**Circuit:**
1. Start with |00⟩
2. Apply Hadamard (H) to qubit 0 → creates superposition
3. Apply CNOT (CX) with control=0, target=1 → creates entanglement

### Task

Create a quantum circuit that generates a Bell state.

In [ ]:
# SOLUTION: Create Bell state circuit

# Create quantum circuit with 2 qubits and 2 classical bits
qc = QuantumCircuit(2, 2)

# Step 1: Apply Hadamard gate to qubit 0
# This creates superposition: |0⟩ → (|0⟩ + |1⟩)/√2
qc.h(0)

# Step 2: Apply CNOT gate (control=0, target=1)
# This creates entanglement
qc.cx(0, 1)

# Step 3: Measure both qubits
qc.measure([0, 1], [0, 1])

print("Bell state circuit created successfully!")
print(f"Circuit has {qc.num_qubits} qubits and {qc.num_clbits} classical bits")
print(f"Circuit depth: {qc.depth()}")
print(f"\nGates used: {[inst.operation.name for inst in qc.data if inst.operation.name != 'measure']}")

### Explanation

**Mathematical evolution:**

1. Initial state: $|00\rangle$

2. After Hadamard on qubit 0:
   $$(H \otimes I)|00\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) \otimes |0\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)$$

3. After CNOT:
   $$CNOT\left(\frac{1}{\sqrt{2}}(|00\rangle + |10\rangle)\right) = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle) = |\Phi^+\rangle$$

The resulting state is **maximally entangled**: measuring qubit 0 instantly determines qubit 1's state.

---

## Exercise 2: Visualise the Circuit

### Background

Visualising circuits helps understand the gate sequence and structure.

### Task

Display the circuit using Qiskit's visualisation tools.

In [ ]:
# SOLUTION: Visualise the circuit

# Method 1: Text representation
print("Circuit diagram (text):")
print(qc.draw(output='text', fold=80))

# Method 2: Matplotlib (graphical)
print("\nCircuit diagram (graphical):")
qc.draw(output='mpl', style='iqp', fold=20)

### Circuit Diagram Interpretation

```
     ┌───┐     ┌─┐
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1
```

**Reading the diagram:**
- `q_0`, `q_1`: Quantum qubits
- `c`: Classical register (stores measurements)
- `H`: Hadamard gate on qubit 0
- `■─X`: CNOT gate (● is control on q0, X is target on q1)
- `M`: Measurement gate
- Vertical lines with `╥`: Measurement results stored in classical bits

---

## Exercise 3: Run Single Simulation

### Background

We use a quantum simulator to execute the circuit and observe measurement outcomes.

In Qiskit 1.x, the workflow is:
1. Create `AerSimulator()` instance
2. Transpile circuit for the simulator
3. Run the transpiled circuit
4. Get results

### Task

Simulate the Bell state circuit once and observe the measurement outcome.

In [ ]:
# SOLUTION: Run single simulation

# Create simulator (Qiskit 1.x)
simulator = AerSimulator()

# Transpile the circuit for the simulator
transpiled_qc = transpile(qc, simulator)

# Run the circuit with 1 shot
job = simulator.run(transpiled_qc, shots=1)
result = job.result()

# Get measurement outcome
counts = result.get_counts()

print("Single measurement result:")
print(f"Outcome: {list(counts.keys())[0]}")
print("\nNote: Due to superposition, each run may give '00' or '11'")
print("We'll never see '01' or '10' due to entanglement!")

### Explanation: Why Only |00⟩ or |11⟩?

The Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ means:

- **50% probability** of measuring |00⟩
- **50% probability** of measuring |11⟩
- **0% probability** of measuring |01⟩ or |10⟩

This is **quantum entanglement**: the qubits are perfectly correlated. When we measure qubit 0:
- If we get 0, qubit 1 will definitely be 0
- If we get 1, qubit 1 will definitely be 1

This correlation is **stronger than any classical correlation**!

---

## Exercise 4: Multiple Circuit Executions (Statistics)

### Background

Since quantum measurement is probabilistic, we need multiple executions (**shots**) to estimate the probability distribution.

**Statistical uncertainty:**
$$\sigma = \sqrt{\frac{p(1-p)}{N}}$$

For 1000 shots and p=0.5: $\sigma \approx 1.6\%$

### Task

Run the circuit 1000 times and analyse the measurement statistics.

In [ ]:
# SOLUTION: Run multiple simulations for statistics

# Set number of shots
num_shots = 1000

# Run simulation
job = simulator.run(transpiled_qc, shots=num_shots)
result = job.result()
counts = result.get_counts()

# Display results
print(f"Measurement results from {num_shots} shots:")
print("\nOutcome counts:")
for outcome, count in sorted(counts.items()):
    percentage = (count / num_shots) * 100
    print(f"  |{outcome}⟩: {count} times ({percentage:.2f}%)")

# Calculate statistics
count_00 = counts.get('00', 0)
count_11 = counts.get('11', 0)
prob_00 = count_00 / num_shots
prob_11 = count_11 / num_shots

# Deviation from theoretical 50-50
deviation = abs(prob_00 - 0.5) * 100

print(f"\nStatistical analysis:")
print(f"  Probability of |00⟩: {prob_00:.4f} (theoretical: 0.5000)")
print(f"  Probability of |11⟩: {prob_11:.4f} (theoretical: 0.5000)")
print(f"  Deviation from 50%: {deviation:.2f}%")
print(f"  Expected uncertainty: ~1.6%")

# Verify entanglement signature
count_01 = counts.get('01', 0)
count_10 = counts.get('10', 0)

if count_01 == 0 and count_10 == 0:
    print("\n✓ Perfect entanglement confirmed: No |01⟩ or |10⟩ observed!")
else:
    print(f"\nNote: {count_01 + count_10} unexpected outcomes (likely simulation noise)")

### Understanding Statistical Fluctuations

**Why not exactly 500/500?**

Quantum measurement is inherently **probabilistic**. Even with a perfect 50-50 probability:
- We expect fluctuations around 500
- Typical range: 480-520 (within ~2 standard deviations)
- Larger deviations are possible but increasingly rare

**This is not an error!** It's the fundamental nature of quantum mechanics.

**To reduce uncertainty:**
- Increase shots: $N = 10,000$ → uncertainty ~0.5%
- But can never eliminate completely (quantum randomness is true randomness)

---

## Exercise 5: Visualise Measurement Distribution

### Task

Create a histogram showing the measurement outcome frequencies.

In [ ]:
# SOLUTION: Visualise measurement distribution

# Use Qiskit's built-in histogram plotter
plot_histogram(counts, figsize=(10, 6), 
               title='Bell State Measurement Outcomes (1000 shots)',
               bar_labels=True)
plt.tight_layout()
plt.show()

print("Histogram interpretation:")
print("- Only |00⟩ and |11⟩ have significant counts")
print("- Both have approximately equal height (~500 each)")
print("- |01⟩ and |10⟩ are absent (or have zero counts)")
print("\nThis is the signature of a Bell state!")

---

## Exercise 6: Multiple Independent Runs

### Background

To understand statistical variation, let's run the experiment multiple times and observe how results vary.

### Task

Run the Bell state circuit 10 times, each with 1000 shots, and analyse consistency.

In [ ]:
# SOLUTION: Multiple independent runs for statistical analysis

num_circuits = 10
num_shots = 1000

# Store results from each run
all_prob_00 = []
all_prob_11 = []
all_deviations = []

print(f"Running {num_circuits} independent experiments...\n")

for i in range(num_circuits):
    # Run circuit
    job = simulator.run(transpiled_qc, shots=num_shots)
    result = job.result()
    counts = result.get_counts()
    
    # Calculate probabilities
    count_00 = counts.get('00', 0)
    count_11 = counts.get('11', 0)
    prob_00 = count_00 / num_shots
    prob_11 = count_11 / num_shots
    
    # Calculate deviation from theoretical 50%
    deviation = abs(prob_00 - 0.5)
    
    # Store results
    all_prob_00.append(prob_00)
    all_prob_11.append(prob_11)
    all_deviations.append(deviation)
    
    # Print result
    print(f"Run {i+1}: |00⟩={count_00:3d} ({prob_00:.3f}), "
          f"|11⟩={count_11:3d} ({prob_11:.3f}), "
          f"Δ={deviation*100:.2f}%")

# Statistical summary
mean_prob_00 = np.mean(all_prob_00)
std_prob_00 = np.std(all_prob_00)
mean_deviation = np.mean(all_deviations)
max_deviation = np.max(all_deviations)

print("\n" + "="*60)
print("Statistical Summary Across All Runs:")
print("="*60)
print(f"Mean P(|00⟩): {mean_prob_00:.4f} (expected: 0.5000)")
print(f"Std Dev:      {std_prob_00:.4f}")
print(f"Mean Deviation: {mean_deviation*100:.2f}%")
print(f"Max Deviation:  {max_deviation*100:.2f}%")
print(f"\nAll deviations within expected range (~1.6%): ",
      "✓" if max_deviation < 0.03 else "Note: Higher than typical")

### Observation: Statistical Consistency

**Key observations:**
1. Each run gives slightly different results (e.g., 487/513, 502/498, etc.)
2. All results cluster around 50% (typically within 48-52%)
3. Average across all runs converges very close to 50%
4. No run produces |01⟩ or |10⟩ outcomes

**This demonstrates:**
- **Individual measurements** have statistical fluctuation
- **Average behavior** matches theoretical prediction
- **Entanglement** is consistently preserved (no forbidden outcomes)

This is **not measurement error** - it's the fundamental quantum nature!

---

## Exercise 7: Enhanced Visualisation

### Task

Create a comprehensive visualisation showing statistical variation across multiple runs.

In [ ]:
# SOLUTION: Enhanced visualisation of statistical variation

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Probability distribution across runs
x = np.arange(1, num_circuits + 1)
width = 0.35

bars1 = ax1.bar(x - width/2, all_prob_00, width, label='P(|00⟩)', 
                color='#1f77b4', edgecolor='black', alpha=0.8)
bars2 = ax1.bar(x + width/2, all_prob_11, width, label='P(|11⟩)', 
                color='#ff7f0e', edgecolor='black', alpha=0.8)

# Add theoretical line
ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=2, 
            label='Theoretical (50%)', alpha=0.7)

# Add error bands (expected range)
ax1.axhspan(0.48, 0.52, alpha=0.1, color='green', 
            label='Expected range (±2%)')

ax1.set_xlabel('Run Number', fontweight='bold')
ax1.set_ylabel('Probability', fontweight='bold')
ax1.set_title('Measurement Probabilities Across Multiple Runs', 
              fontweight='bold', fontsize=14)
ax1.legend(loc='upper right')
ax1.set_xticks(x)
ax1.set_ylim(0.40, 0.60)
ax1.grid(True, alpha=0.3)

# Plot 2: Deviation distribution (histogram)
deviations_percent = [d * 100 for d in all_deviations]
ax2.hist(deviations_percent, bins=10, color='#2ca02c', 
         edgecolor='black', alpha=0.7)
ax2.axvline(np.mean(deviations_percent), color='red', 
            linestyle='--', linewidth=2,
            label=f'Mean: {np.mean(deviations_percent):.2f}%')

ax2.set_xlabel('Deviation from 50% (%)', fontweight='bold')
ax2.set_ylabel('Frequency', fontweight='bold')
ax2.set_title('Distribution of Statistical Deviations', 
              fontweight='bold', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualisation interpretation:")
print("Left plot:  Probabilities fluctuate around 50% (red line)")
print("            Most points fall within green band (±2%)")
print("Right plot: Distribution of deviations (typical: 1-2%)")
print("\nConclusion: Statistical variation is expected and normal!")

---

## Exercise 8: Comparison with Separable State

### Background

To appreciate entanglement, let's compare with a **separable (non-entangled)** state.

**Separable state:** $|+0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) \otimes |0\rangle$

This has:
- Qubit 0 in superposition
- Qubit 1 always in |0⟩
- No entanglement!

### Task

Create a separable state circuit and compare outcomes with Bell state.

In [ ]:
# SOLUTION: Create and simulate separable state

# Create circuit with only Hadamard (no CNOT = no entanglement)
qc_separable = QuantumCircuit(2, 2)
qc_separable.h(0)  # Only superposition on qubit 0
qc_separable.measure([0, 1], [0, 1])

print("Separable state circuit:")
print(qc_separable.draw(output='text'))

# Simulate
transpiled_sep = transpile(qc_separable, simulator)
job = simulator.run(transpiled_sep, shots=1000)
result = job.result()
counts_sep = result.get_counts()

# Display results
print("\nSeparable state results:")
for outcome, count in sorted(counts_sep.items()):
    prob = count / 1000
    print(f"  |{outcome}⟩: {count} ({prob:.3f})")

# Compare with Bell state
print("\n" + "="*60)
print("Comparison: Bell State vs Separable State")
print("="*60)
print("\nBell State (Entangled):")
print("  Outcomes: Only |00⟩ and |11⟩ (perfectly correlated)")
print("  P(|00⟩) ≈ 50%, P(|11⟩) ≈ 50%")
print("\nSeparable State (Not Entangled):")
print("  Outcomes: |00⟩ and |10⟩ (qubit 1 always 0)")
print("  P(|00⟩) ≈ 50%, P(|10⟩) ≈ 50%")
print("\nKey Difference:")
print("  Entangled:   Qubits are correlated (both 0 OR both 1)")
print("  Separable:   Qubits are independent (q0 random, q1 always 0)")

# Visualise comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bell state (from earlier)
job_bell = simulator.run(transpiled_qc, shots=1000)
counts_bell = job_bell.result().get_counts()
states = ['00', '01', '10', '11']
bell_counts = [counts_bell.get(s, 0) for s in states]

ax1.bar(states, bell_counts, color='#1f77b4', edgecolor='black', alpha=0.7)
ax1.set_title('Bell State (Entangled)', fontweight='bold', fontsize=14)
ax1.set_ylabel('Counts', fontweight='bold')
ax1.set_xlabel('Measurement Outcome', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Separable state
sep_counts = [counts_sep.get(s, 0) for s in states]
ax2.bar(states, sep_counts, color='#ff7f0e', edgecolor='black', alpha=0.7)
ax2.set_title('Separable State (Not Entangled)', fontweight='bold', fontsize=14)
ax2.set_ylabel('Counts', fontweight='bold')
ax2.set_xlabel('Measurement Outcome', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Understanding the Difference

**Bell State (Entangled):**
$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$
- Qubits are **correlated**: If q0=0 then q1=0, if q0=1 then q1=1
- This correlation exists **even if qubits are separated by vast distances**
- Cannot be described as independent qubits

**Separable State:**
$$|+0\rangle = \left(\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)\right) \otimes |0\rangle$$
- Qubit 0 is in superposition (random outcome)
- Qubit 1 is always |0⟩
- Can be described as **product** of independent qubit states
- No correlation between qubits

**This distinction is fundamental to quantum computing!**

---

## Summary and Key Takeaways

### ✅ Concepts Mastered

1. **Quantum Circuit Creation** - Building circuits with Qiskit 1.x
2. **Bell States** - Maximally entangled two-qubit states
3. **Entanglement** - Quantum correlations stronger than classical
4. **Measurement** - Probabilistic collapse of superposition
5. **Statistics** - Understanding quantum measurement uncertainty
6. **Qiskit 1.x API** - Modern patterns for quantum programming

### 🔑 Key Insights

| Concept | Key Point |
|---------|----------|
| **Superposition** | Qubits can be in multiple states simultaneously |
| **Entanglement** | Qubits become correlated in non-classical ways |
| **Measurement** | Collapses superposition probabilistically |
| **Bell State** | $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ - perfect correlation |
| **Statistics** | Need many shots to estimate probabilities |
| **Uncertainty** | $\sigma \propto 1/\sqrt{N}$ - decreases with more shots |

### 📊 Experimental Results

From our simulations:
- Bell states consistently show only |00⟩ and |11⟩ outcomes
- Probabilities cluster around 50% each with ~1-2% deviation
- No |01⟩ or |10⟩ observed (signature of entanglement)
- Statistical variation is normal and expected

### 🎯 What's Next?

1. **Task 3** - Use entanglement for quantum teleportation
2. **Task 5** - Multi-qubit entanglement for error correction
3. **Advanced topics:**
   - Other Bell states (Φ⁻, Ψ±)
   - GHZ states (3-qubit entanglement)
   - Bell inequality violations
   - Quantum dense coding

### 💡 Real-World Applications

Entanglement enables:
- **Quantum teleportation** (Task 3)
- **Quantum cryptography** (QKD)
- **Quantum error correction** (Task 5)
- **Quantum algorithms** (Shor's, Grover's)

**Congratulations!** You've successfully created and analysed quantum entanglement using Qiskit 1.x! 🎉